In [3]:
nav = nav.sort_values(

["amfi_code","date"]

)

nav["daily_return"] = nav.groupby(

"amfi_code"

)["nav"].pct_change()

In [4]:
cvar=[]

for code,group in nav.groupby("amfi_code"):

    ret=group["daily_return"].dropna()

    var=np.percentile(ret,5)

    cvar_val=ret[ret<=var].mean()

    cvar.append(

        [code,cvar_val]

    )

cvar_df=pd.DataFrame(

cvar,

columns=[

"amfi_code",

"CVaR"

]

)

cvar_df.head()

,amfi_code,CVaR
0,100016,-0.018060
1,100025,-0.004994
2,100033,-0.023456
3,101206,-0.017439
4,101207,-0.032459


In [6]:
var_list=[]

for code, group in nav.groupby("amfi_code"):

    ret = group["daily_return"].dropna()

    var95 = np.percentile(ret,5)

    var_list.append([code,var95])

var_df = pd.DataFrame(
    var_list,
    columns=["amfi_code","VaR_95"]
)

var_df.head()

,amfi_code,VaR_95
0,100016,-0.014364
1,100025,-0.003793
2,100033,-0.019034
3,101206,-0.013282
4,101207,-0.026021


In [7]:
risk=var_df.merge(

cvar_df,

on="amfi_code"

)

risk.to_csv(

"../reports/var_cvar_report.csv",

index=False

)

In [8]:
rf=0.065

nav["rolling_sharpe"]=(

nav.groupby("amfi_code")

["daily_return"]

.rolling(90)

.mean()

.reset_index(level=0,drop=True)

/

nav.groupby("amfi_code")

["daily_return"]

.rolling(90)

.std()

.reset_index(level=0,drop=True)

)*np.sqrt(252)

In [9]:
top5=performance.sort_values(

"aum_crore",

ascending=False

).head(5)

codes=top5["amfi_code"]

plot_df=nav[

nav["amfi_code"]

.isin(codes)

]

import plotly.express as px

fig=px.line(

plot_df,

x="date",

y="rolling_sharpe",

color="amfi_code",

title="Rolling 90 Day Sharpe"

)

fig.show()

In [10]:
fig.write_image(

"../reports/rolling_sharpe_chart.png"

)

In [11]:
transactions["cohort_year"]=(
transactions

.groupby("investor_id")

["transaction_date"]

.transform("min")

.dt.year
)

cohort=transactions.groupby(

"cohort_year"

).agg(

sip_amount=("amount_inr","sum"),

investors=("investor_id","nunique")

)

cohort

,sip_amount,investors
cohort_year,,
2024,3491125187,4803
2025,30455243,197


In [13]:
transactions = transactions.sort_values(
    ["investor_id", "transaction_date"]
)

transactions["gap"] = (
    transactions.groupby("investor_id")["transaction_date"]
    .diff()
    .dt.days
)

risk_investors = transactions[
    transactions["gap"] > 35
]

risk_investors.head()

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,cohort_year,gap
24079,INV000001,2025-01-14,148569,LUMPSUM,189483,Haryana,Gurugram,T30,36-45,Male,19.9,UPI,Verified,2024,71.0
12522,INV000002,2024-07-14,149323,LUMPSUM,153187,Maharashtra,Pune,T30,46-55,Male,24.0,UPI,Verified,2024,107.0
16803,INV000002,2024-09-21,120841,SIP,2354,Maharashtra,Pune,T30,46-55,Male,24.0,Mandate,Verified,2024,69.0
24661,INV000002,2025-01-23,118632,LUMPSUM,317170,Maharashtra,Pune,T30,46-55,Male,24.0,Mandate,Verified,2024,112.0
31881,INV000002,2025-05-17,119094,SIP,2690,Maharashtra,Pune,T30,46-55,Male,24.0,Cheque,Verified,2024,114.0


In [15]:
hhi = holdings.groupby(
    "amfi_code"
)["weight_pct"].apply(
    lambda x: ((x / 100) ** 2).sum()
)

hhi.head()

amfi_code
100016    0.139534
100033    0.147592
101206    0.129332
101207    0.200700
102885    0.174709
Name: weight_pct, dtype: float64

1. Funds with highest Sharpe ratios outperform peers consistently.

2. Investors from earlier cohorts invest larger amounts.

3. SIP gaps >35 days indicate at-risk investors.

4. Low HHI funds are more diversified.

5. VaR and CVaR identify downside risk clearly.